# Exercise 1: Taboo in Kazakh

**Level: introductory worked exercise.** The “Your turn” task is an extension.

*Generation and Dialogue · version 0.1.0*

> Play Kazakh Taboo with a model as describer and guesser, using the controller and rule checker of the Hands-On. Compare the surface and lemma checkers, track the language of each clue by round, and decide how to score inflected guesses. Then document every language-specific decision needed to port one more clembench game.

**Setup.** `pip install llama-cpp-python fasttext-wheel requests "turkicnlp[hfst]==0.5.0"` (the compiled `hfst` package needs Python 3.11 or 3.12). The generator is **Qwen2.5-1.5B-Instruct** in 8-bit GGUF form, run on the CPU with llama.cpp: set `QWEN_GGUF` to the file. The book's run converted the Hugging Face checkpoint at revision `989aa79` with llama.cpp's `convert_hf_to_gguf.py --outtype q8_0` (SHA-256 recorded in the outputs); Qwen's own `Qwen2.5-1.5B-Instruct-GGUF` q8_0 file is an equivalent starting point. Language identification uses GlotLID v3 (`GLOTLID_PATH`, otherwise downloaded from the Hub), morphology the Apertium analysers that come with TurkicNLP. Decoding is greedy (temperature 0) and every model reply is cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`), so the outputs can be inspected and a rerun is exact. **One small model is used throughout.** Its errors illustrate the measurement methods; they are not a ranking of models or a statement about larger ones.

The game uses `experiments/n3_generation/game.py` (bounded controller; the guesser never sees the target) and `turkic_taboo_checker.py` (surface hits, lemma candidates from all Apertium readings, unresolved tokens). Each game is played twice, once with each checker deciding whether a clue is valid. The twelve targets and forbidden lemmas were chosen for this exercise and should be checked by a speaker.

In [ ]:
import collections, hashlib, json, math, os, random, re, statistics, unicodedata
from pathlib import Path
import requests

CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
THREADS = int(os.environ.get("THREADS", os.cpu_count() or 1))
QWEN_GGUF = os.environ.get("QWEN_GGUF", str(CACHE / "models/qwen2.5-1.5b-instruct-q8_0.gguf"))
SIB = "https://raw.githubusercontent.com/dadelani/sib-200/efad32e0fc601cfbfdf5ca80672f29697f6f401f/data/annotated/"
KAZNERD = "https://raw.githubusercontent.com/IS2AI/KazNERD/bd4333d0f5952b9fafb2ef2ac2fefa0ad3c0333f/KazNERD/IOB2_test.txt"
FETCHED = {}

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300); r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content)
    FETCHED[url] = hashlib.sha256(path.read_bytes()).hexdigest()
    return path

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

# ---- the generator, with every reply cached ---------------------------------------------------
_LLM = {}
REPLIES = CACHE / "ch14-replies.jsonl"
_CACHED = {}
if REPLIES.exists():
    for line in REPLIES.read_text(encoding="utf-8").splitlines():
        r = json.loads(line); _CACHED[r["key"]] = r["reply"]

def chat(messages, max_tokens=200):
    """Greedy reply of Qwen2.5-1.5B-Instruct (llama.cpp, q8_0) to a list of chat messages."""
    key = hashlib.sha1(json.dumps([messages, max_tokens], ensure_ascii=False).encode()).hexdigest()
    if key not in _CACHED:
        if "m" not in _LLM:
            from llama_cpp import Llama
            _LLM["m"] = Llama(model_path=QWEN_GGUF, n_ctx=4096, n_threads=THREADS, seed=0, verbose=False)
        out = _LLM["m"].create_chat_completion(messages=messages, temperature=0.0, max_tokens=max_tokens)
        _CACHED[key] = out["choices"][0]["message"]["content"].strip()
        REPLIES.parent.mkdir(parents=True, exist_ok=True)
        with REPLIES.open("a", encoding="utf-8") as f:
            f.write(json.dumps({"key": key, "messages": messages, "reply": _CACHED[key]}, ensure_ascii=False) + "\n")
    return _CACHED[key]

# ---- automatic fidelity signals ---------------------------------------------------------------
_LID = {}
def lid(text):
    """Unrestricted GlotLID v3 label and probability of one line of text."""
    if "m" not in _LID:
        import fasttext
        path = os.environ.get("GLOTLID_PATH")
        if not path:
            from huggingface_hub import hf_hub_download
            path = hf_hub_download("cis-lmu/glotlid", "model_v3.bin")
        _LID["m"] = fasttext.load_model(path)
    labels, probs = _LID["m"].predict([text.replace("\n", " ")], k=1)
    return labels[0][0].replace("__label__", ""), float(probs[0][0])

SENT = re.compile(r"(?<=[.!?…])\s+|\n+")
def sentences(text):
    return [s.strip() for s in SENT.split(text) if len(re.findall(r"[^\W\d_]", s)) >= 3]

def script_shares(text):
    letters = [c for c in text if c.isalpha()]
    count = collections.Counter("Cyrl" if "CYRILLIC" in unicodedata.name(c, "") else
                                "Latn" if "LATIN" in unicodedata.name(c, "") else "other" for c in letters)
    return {k: v / len(letters) for k, v in count.items()} if letters else {}

WORD = re.compile(r"[^\W\d_]+")
_FST = {}
def lookup(lang):
    """Raw Apertium analyser of TurkicNLP for kaz (Cyrillic) or uzb (Latin): word -> [(reading, weight)]."""
    if lang not in _FST:
        import hfst, turkicnlp
        from turkicnlp.resources.registry import ModelRegistry
        script = {"kaz": "Cyrl", "uzb": "Latn"}[lang]
        turkicnlp.download(lang, processors=["morph"])
        path = ModelRegistry.get_model_path(lang, "morph", "apertium", script=script) / f"{lang}.automorf.hfst"
        fst = hfst.HfstInputStream(str(path)).read()
        _FST[lang] = (lambda w: fst.lookup(w) or fst.lookup(w.lower())), sha256(path)
    return _FST[lang][0]

def accepted(word, lang):
    return any(r and not r.startswith("*") and math.isfinite(w) for r, w in lookup(lang)(word))

def coverage(texts, lang):
    words = [w for t in texts for w in WORD.findall(t)]
    ok = sum(accepted(w, lang) for w in words)
    return ok, len(words)

print("generator:", Path(QWEN_GGUF).name, "| sha256", sha256(QWEN_GGUF)[:16] if Path(QWEN_GGUF).exists() else "missing")

In [ ]:
import sys
sys.path.insert(0, str(Path("../experiments/n3_generation").resolve()))
from game import play
from turkic_taboo_checker import check_tokens
# Twelve everyday targets, each with three related forbidden lemmas in Apertium's convention
# (nouns in the nominative, verbs as bare stems). Chosen for this exercise; have a speaker check them.
TARGETS = {"кітап": ["оқы", "бет", "жаз"], "мектеп": ["оқушы", "мұғалім", "сабақ"], "нан": ["пісір", "ұн", "тамақ"],
           "су": ["іш", "өзен", "сұйық"], "жылқы": ["мін", "ат", "мал"], "тау": ["биік", "шың", "тас"],
           "қала": ["көше", "тұрғын", "үй"], "дәрігер": ["аурухана", "емде", "науқас"], "күн": ["жарық", "аспан", "ыстық"],
           "шай": ["іш", "кесе", "сүт"], "қар": ["қыс", "ақ", "суық"], "балық": ["су", "аула", "өзен"]}
look = lookup("kaz")
def raw(token):
    return [(r, w) for r, w in look(token)]
def describe_fn(target, forbidden):
    def describe(history):
        msgs = [{"role": "system", "content": f"You are playing Taboo in Kazakh. Describe the Kazakh word '{target}' in one or two "
                 f"short Kazakh sentences so that a partner can guess it. Do not use the word itself, the words {', '.join(forbidden)}, "
                 "or any of their forms. Write only Kazakh in Cyrillic script."}]
        for h in history[1:]:
            msgs.append({"role": "assistant" if h["role"] == "assistant" else "user", "content": h["content"]})
        if len(msgs) == 1:
            msgs.append({"role": "user", "content": "Give your first clue."})
        return chat(msgs, max_tokens=80)
    return describe
def guess(history):
    msgs = [{"role": "system", "content": "You are playing Taboo in Kazakh. Your partner describes one Kazakh word. "
             "Answer with that single Kazakh word only, in Cyrillic, without explanation."}]
    msgs += [{"role": "user" if h["role"] == "user" else "assistant", "content": h["content"]} for h in history[1:]]
    return chat(msgs, max_tokens=10)
def lemmas(word):
    return {r.split("<", 1)[0].lower() for r, w in look(word) if "<" in r} | {word.lower()}
def matches(answer, target):
    words = WORD.findall(answer.lower())
    return bool(words) and target in lemmas(words[0])            # lemma match: an inflected guess counts

games, report = {}, collections.Counter()
for checker in ("surface", "lemma"):
    for target, forbidden in TARGETS.items():
        banned = [target] + forbidden
        def check(clue, banned=banned):
            r = check_tokens(WORD.findall(clue), banned, raw, "kaz")
            if r["surface_hits"]:
                return "invalid"
            return "invalid" if checker == "lemma" and r["lemma_candidates"] else "valid"
        games[checker, target] = play(target, banned, describe_fn(target, forbidden), guess, check, matches, rounds=3)
print("outcomes of 12 Kazakh Taboo games (Qwen2.5-1.5B-Instruct as describer and guesser, at most 3 rounds):")
for checker in ("surface", "lemma"):
    print(f"  {checker:<8} checker:", dict(collections.Counter(g["outcome"] for (c, _), g in games.items() if c == checker)))
clues = [(t, turn["clue"]) for (c, t), g in games.items() if c == "surface" for turn in g["turns"]]
hidden = []
for t, clue in clues:
    r = check_tokens(WORD.findall(clue), [t] + TARGETS[t], raw, "kaz")
    if not r["surface_hits"] and r["lemma_candidates"]:
        hidden.append((t, r["lemma_candidates"]))
surface_hits = sum(bool(check_tokens(WORD.findall(c), [t] + TARGETS[t], raw, "kaz")["surface_hits"]) for t, c in clues)
print(f"clues produced under the surface checker: {len(clues)}; with a surface hit: {surface_hits}; "
      f"with only an inflected forbidden word (missed by the surface checker): {len(hidden)}")
for t, cand in hidden[:6]:
    print(f"  target {t}: {cand}")
unres = [u for t, c in clues for u in check_tokens(WORD.findall(c), [t] + TARGETS[t], raw, "kaz")["unresolved"]]
print(f"tokens without any analyser reading (left to a human): {len(unres)} of {sum(len(WORD.findall(c)) for _, c in clues)}, e.g. {unres[:8]}")
print("language of each clue (GlotLID):", dict(collections.Counter(lid(c)[0] for _, c in clues)))
by_round = collections.defaultdict(collections.Counter)
for (c, t), g in games.items():
    if c == "surface":
        for turn in g["turns"]:
            by_round[turn["round"]][lid(turn["clue"])[0]] += 1
print("clue language by round:", {r: dict(v) for r, v in sorted(by_round.items())})
gs = [(t, turn["guess"]) for (c, t), g in games.items() if c == "surface" for turn in g["turns"] if "guess" in turn]
kinds = collections.Counter("exact" if a.strip(" .!").lower() == t else "inflected target" if matches(a, t)
                            else "one word" if len(WORD.findall(a)) == 1 else "several words" for t, a in gs)
print(f"guesses: {len(gs)}; {dict(kinds)}; examples: {gs[:8]}")
cyr = [script_shares(c).get("Cyrl", 0) for _, c in clues]
print(f"mean Cyrillic share of clue letters: {statistics.mean(cyr):.1%}; clues below 90% Cyrillic: {sum(x < 0.9 for x in cyr)}")
g = games["lemma", "кітап"]
print("example game (lemma checker, target кітап):")
for turn in g["turns"]:
    print(f"  round {turn['round']}: clue: {turn['clue'][:150]}\n           rule: {turn['rule_status']}  guess: {turn.get('guess', '-')}")
print("  outcome:", g["outcome"])

## Worked solution

Output of `experiments/ch14_exercises/run_ch14_exercises.py`:

```
outcomes of 12 Kazakh Taboo games (Qwen2.5-1.5B-Instruct as describer and guesser, at most 3 rounds):
  surface  checker: {'invalid': 3, 'turn_limit': 9}
  lemma    checker: {'invalid': 3, 'turn_limit': 9}
clues produced under the surface checker: 30; with a surface hit: 3; with only an inflected forbidden word (missed by the surface checker): 0
tokens without any analyser reading (left to a human): 72 of 246, e.g. ['бул', 'сөздөрдүн', 'аныктаныктын', 'жазуу', 'өзгөчөлік', 'білімдік', 'салымты', 'білімдік']
language of each clue (GlotLID): {'kir_Cyrl': 4, 'kaz_Cyrl': 26}
clue language by round: {1: {'kir_Cyrl': 2, 'kaz_Cyrl': 10}, 2: {'kaz_Cyrl': 8, 'kir_Cyrl': 1}, 3: {'kaz_Cyrl': 8, 'kir_Cyrl': 1}}
guesses: 27; {'one word': 24, 'several words': 3}; examples: [('мектеп', 'Бұлар'), ('мектеп', 'Бұлар'), ('мектеп', 'Бұлар'), ('нан', 'Көңіл'), ('нан', 'Көңілді'), ('нан', 'Көңілдің'), ('су', 'Бұл құрылым.'), ('су', 'Бұл құрылым.')]
mean Cyrillic share of clue letters: 95.0%; clues below 90% Cyrillic: 3
example game (lemma checker, target кітап):
  round 1: clue: Кітап - бул сөздөрдүн аныктаныктын жазуу.
           rule: invalid  guess: -
  outcome: invalid
```

**No game is won.** Three of twelve games stop because the opening clue contains the target; nine reach the three-round limit. The surface and lemma checkers give the same outcomes. Of 27 guesses, 24 are single words and three are phrases. The implemented matcher examines only the first word, which should be documented when scoring multiword replies.

**No additional lemma violations are detected.** This does not mean none were missed. There are 72 unresolved words among 246 clue words, and this exercise lets them pass. The Hands-On instead calls for review. The target list and ambiguous readings still need a speaker's judgement.

**Language labels are diagnostics.** GlotLID labels four of 30 clues Kyrgyz: two in round one and one in each later round. Three clues have less than 90% Cyrillic letters. This small sample does not establish a trend over turns.

**Porting another game.** For Wordle, specify the alphabet, allowed words, letter-counting rules, casing and Unicode normalisation. Validate word familiarity and difficulty with speakers before comparing languages.

## Your turn — extension

1. Replace the target list with one chosen by a speaker across frequency bands, and add a Turkish control.
2. Change `check` so that a clue with an unresolved token returns `review`, and count how many games stop for review.